# 02 silver transform - clean bronze tables, bad rows go to quarantine


In [ ]:
f
r
o
m
 
p
y
s
p
a
r
k
.
s
q
l
 
i
m
p
o
r
t
 
f
u
n
c
t
i
o
n
s
 
a
s
 
F
,
 
W
i
n
d
o
w

f
r
o
m
 
d
e
l
t
a
.
t
a
b
l
e
s
 
i
m
p
o
r
t
 
D
e
l
t
a
T
a
b
l
e
 
 
s
h
i
p
s
 
w
i
t
h
 
t
h
e
 
F
a
b
r
i
c
 
S
p
a
r
k
 
r
u
n
t
i
m
e


b
a
t
c
h
_
i
d
 
=
 
N
o
n
e
 
 
p
a
r
a
m
e
t
e
r
:
 
n
o
r
m
a
l
l
y
 
p
a
s
s
e
d
 
f
r
o
m
 
t
h
e
 
p
i
p
e
l
i
n
e

T
A
B
L
E
S
 
=
 
(
"
a
b
f
s
s
:
/
/
<
W
O
R
K
S
P
A
C
E
_
N
A
M
E
>
@
o
n
e
l
a
k
e
.
d
f
s
.
f
a
b
r
i
c
.
m
i
c
r
o
s
o
f
t
.
c
o
m
/
"

 
"
<
L
A
K
E
H
O
U
S
E
_
N
A
M
E
>
.
L
a
k
e
h
o
u
s
e
/
T
a
b
l
e
s
"
)
 
 
T
O
D
O
:
 
y
o
u
r
 
l
a
k
e
h
o
u
s
e


d
e
f
 
b
r
o
n
z
e
(
n
a
m
e
)
:

 
d
f
 
=
 
s
p
a
r
k
.
r
e
a
d
.
f
o
r
m
a
t
(
"
d
e
l
t
a
"
)
.
l
o
a
d
(
f
"
{
T
A
B
L
E
S
}
/
b
r
o
n
z
e
_
{
n
a
m
e
}
"
)

 
r
e
t
u
r
n
 
d
f
 
i
f
 
b
a
t
c
h
_
i
d
 
i
s
 
N
o
n
e
 
e
l
s
e
 
d
f
.
f
i
l
t
e
r
(
F
.
c
o
l
(
"
_
b
a
t
c
h
_
i
d
"
)
 
=
=
 
b
a
t
c
h
_
i
d
)


d
e
f
 
w
r
i
t
e
_
d
e
l
t
a
(
d
f
,
 
n
a
m
e
,
 
m
o
d
e
=
"
o
v
e
r
w
r
i
t
e
"
)
:

 
d
f
.
w
r
i
t
e
.
f
o
r
m
a
t
(
"
d
e
l
t
a
"
)
.
m
o
d
e
(
m
o
d
e
)
.
s
a
v
e
(
f
"
{
T
A
B
L
E
S
}
/
{
n
a
m
e
}
"
)


In [ ]:
d
e
f
 
s
t
d
_
c
o
d
e
(
c
o
l
)
:
 
 
'
 
p
0
1
 
'
 
/
 
'
P
0
1
'
 
/
 
'
p
0
1
'
 
a
l
l
 
b
e
c
o
m
e
 
'
P
0
1
'

 
r
e
t
u
r
n
 
F
.
u
p
p
e
r
(
F
.
t
r
i
m
(
c
o
l
)
)


d
e
f
 
s
t
d
_
n
a
m
e
(
c
o
l
)
:
 
 
c
o
l
l
a
p
s
e
 
i
n
t
e
r
n
a
l
 
w
h
i
t
e
s
p
a
c
e
,
 
t
i
t
l
e
-
c
a
s
e
 
d
i
s
p
l
a
y
 
n
a
m
e
s

 
r
e
t
u
r
n
 
F
.
i
n
i
t
c
a
p
(
F
.
r
e
g
e
x
p
_
r
e
p
l
a
c
e
(
F
.
t
r
i
m
(
c
o
l
)
,
 
r
"
\
s
+
"
,
 
"
 
"
)
)


d
e
f
 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
c
o
l
)
:

 
"
"
"
S
o
u
r
c
e
s
 
d
i
s
a
g
r
e
e
 
o
n
 
d
a
t
e
 
f
o
r
m
a
t
.
 
T
r
y
 
e
a
c
h
 
k
n
o
w
n
 
p
a
t
t
e
r
n
;
 
u
n
p
a
r
s
e
a
b
l
e

 
v
a
l
u
e
s
 
b
e
c
o
m
e
 
n
u
l
l
 
H
E
R
E
 
s
o
 
t
h
e
 
n
u
l
l
 
c
h
e
c
k
s
 
b
e
l
o
w
 
c
a
n
 
q
u
a
r
a
n
t
i
n
e
 
t
h
e
m
.
"
"
"

 
r
e
t
u
r
n
 
F
.
c
o
a
l
e
s
c
e
(

 
F
.
t
o
_
d
a
t
e
(
c
o
l
,
 
"
y
y
y
y
-
M
M
-
d
d
"
)
,
 
F
.
t
o
_
d
a
t
e
(
c
o
l
,
 
"
M
M
/
d
d
/
y
y
y
y
"
)
,

 
F
.
t
o
_
d
a
t
e
(
c
o
l
,
 
"
d
d
-
M
M
M
-
y
y
"
)
,
 
 
F
.
t
o
_
d
a
t
e
(
c
o
l
,
 
"
d
d
-
M
M
M
-
y
y
y
y
"
)
,

 
F
.
t
o
_
d
a
t
e
(
c
o
l
,
 
"
y
y
y
y
/
M
M
/
d
d
"
)
)


d
e
f
 
d
e
d
u
p
e
(
d
f
,
 
n
a
t
u
r
a
l
_
k
e
y
s
,
 
o
r
d
e
r
_
c
o
l
=
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
:

 
"
"
"
K
e
e
p
 
e
x
a
c
t
l
y
 
o
n
e
 
r
o
w
 
p
e
r
 
n
a
t
u
r
a
l
 
k
e
y
 
—
 
t
h
e
 
m
o
s
t
 
r
e
c
e
n
t
l
y
 
i
n
g
e
s
t
e
d
 
c
o
p
y
.

 
S
o
u
r
c
e
 
s
y
s
t
e
m
s
 
r
e
-
s
e
n
d
 
c
o
r
r
e
c
t
e
d
 
r
o
w
s
;
 
B
r
o
n
z
e
 
k
e
e
p
s
 
t
h
e
m
 
a
l
l
,
 
S
i
l
v
e
r
 
p
i
c
k
s
.
"
"
"

 
w
 
=
 
W
i
n
d
o
w
.
p
a
r
t
i
t
i
o
n
B
y
(
*
n
a
t
u
r
a
l
_
k
e
y
s
)
.
o
r
d
e
r
B
y
(
F
.
c
o
l
(
o
r
d
e
r
_
c
o
l
)
.
d
e
s
c
(
)
)

 
r
e
t
u
r
n
 
(
d
f
.
w
i
t
h
C
o
l
u
m
n
(
"
_
r
n
"
,
 
F
.
r
o
w
_
n
u
m
b
e
r
(
)
.
o
v
e
r
(
w
)
)

 
.
f
i
l
t
e
r
(
F
.
c
o
l
(
"
_
r
n
"
)
 
=
=
 
1
)
.
d
r
o
p
(
"
_
r
n
"
)
)


_
q
u
a
r
a
n
t
i
n
e
_
r
o
w
s
 
=
 
[
]

d
e
f
 
q
u
a
r
a
n
t
i
n
e
(
d
f
,
 
r
e
a
s
o
n
)
:

 
"
"
"
C
o
l
l
e
c
t
 
r
e
j
e
c
t
e
d
 
r
o
w
s
 
w
i
t
h
 
t
h
e
 
r
u
l
e
 
t
h
a
t
 
r
e
j
e
c
t
e
d
 
t
h
e
m
.
 
W
r
i
t
t
e
n
 
o
n
c
e
 
a
t

 
t
h
e
 
e
n
d
 
s
o
 
q
u
a
r
a
n
t
i
n
e
 
i
s
 
a
 
s
i
n
g
l
e
 
a
u
d
i
t
a
b
l
e
 
t
a
b
l
e
,
 
n
o
t
 
s
c
a
t
t
e
r
e
d
 
d
e
l
e
t
e
s
.
"
"
"

 
_
q
u
a
r
a
n
t
i
n
e
_
r
o
w
s
.
a
p
p
e
n
d
(

 
d
f
.
w
i
t
h
C
o
l
u
m
n
(
"
_
q
u
a
r
a
n
t
i
n
e
_
r
e
a
s
o
n
"
,
 
F
.
l
i
t
(
r
e
a
s
o
n
)
)

 
.
w
i
t
h
C
o
l
u
m
n
(
"
_
q
u
a
r
a
n
t
i
n
e
d
_
a
t
"
,
 
F
.
c
u
r
r
e
n
t
_
t
i
m
e
s
t
a
m
p
(
)
)
)


d
e
f
 
s
p
l
i
t
_
v
a
l
i
d
(
d
f
,
 
r
u
l
e
,
 
r
e
a
s
o
n
)
:

 
"
"
"
R
e
t
u
r
n
s
 
r
o
w
s
 
p
a
s
s
i
n
g
 
`
r
u
l
e
`
;
 
s
e
n
d
s
 
t
h
e
 
r
e
s
t
 
t
o
 
q
u
a
r
a
n
t
i
n
e
.
"
"
"

 
q
u
a
r
a
n
t
i
n
e
(
d
f
.
f
i
l
t
e
r
(
~
r
u
l
e
)
,
 
r
e
a
s
o
n
)

 
r
e
t
u
r
n
 
d
f
.
f
i
l
t
e
r
(
r
u
l
e
)


In [ ]:
 
-
-
-
 
D
i
m
P
l
a
n
t
 
/
 
D
i
m
M
a
c
h
i
n
e
 
/
 
D
i
m
R
e
t
a
i
l
e
r
:
 
c
l
e
a
n
,
 
d
e
d
u
p
e
,
 
s
u
r
r
o
g
a
t
e
 
k
e
y
 
-
-
-
-
-
-
-

d
i
m
_
p
l
a
n
t
 
=
 
d
e
d
u
p
e
(

 
b
r
o
n
z
e
(
"
p
l
a
n
t
s
"
)
.
s
e
l
e
c
t
(

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
l
a
n
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
l
a
n
t
_
i
d
"
)
,

 
s
t
d
_
n
a
m
e
(
F
.
c
o
l
(
"
p
l
a
n
t
_
n
a
m
e
"
)
)
.
a
l
i
a
s
(
"
p
l
a
n
t
_
n
a
m
e
"
)
,

 
s
t
d
_
n
a
m
e
(
F
.
c
o
l
(
"
c
i
t
y
"
)
)
.
a
l
i
a
s
(
"
c
i
t
y
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
s
t
a
t
e
"
)
)
.
a
l
i
a
s
(
"
s
t
a
t
e
"
)
,

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
t
i
m
e
z
o
n
e
"
)
)
.
a
l
i
a
s
(
"
t
i
m
e
z
o
n
e
"
)
,

 
F
.
c
o
l
(
"
o
p
e
n
e
d
_
y
e
a
r
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
o
p
e
n
e
d
_
y
e
a
r
"
)
,

 
F
.
c
o
l
(
"
_
b
a
t
c
h
_
i
d
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
s
y
s
t
e
m
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
f
i
l
e
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)
,
 
[
"
p
l
a
n
t
_
i
d
"
]
)

d
i
m
_
p
l
a
n
t
 
=
 
(
d
i
m
_
p
l
a
n
t
.
w
i
t
h
C
o
l
u
m
n
(
"
p
l
a
n
t
_
k
e
y
"
,
 
F
.
m
d
5
(
F
.
c
o
l
(
"
p
l
a
n
t
_
i
d
"
)
)
)

 
.
s
e
l
e
c
t
(
"
p
l
a
n
t
_
k
e
y
"
,
 
"
p
l
a
n
t
_
i
d
"
,
 
"
p
l
a
n
t
_
n
a
m
e
"
,
 
"
c
i
t
y
"
,
 
"
s
t
a
t
e
"
,

 
"
t
i
m
e
z
o
n
e
"
,
 
"
o
p
e
n
e
d
_
y
e
a
r
"
)
)

p
l
a
n
t
_
k
e
y
s
 
=
 
d
i
m
_
p
l
a
n
t
.
s
e
l
e
c
t
(
"
p
l
a
n
t
_
i
d
"
,
 
"
p
l
a
n
t
_
k
e
y
"
)


d
i
m
_
m
a
c
h
i
n
e
 
=
 
d
e
d
u
p
e
(

 
b
r
o
n
z
e
(
"
m
a
c
h
i
n
e
s
"
)
.
s
e
l
e
c
t
(

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
m
a
c
h
i
n
e
_
i
d
"
)
)
.
a
l
i
a
s
(
"
m
a
c
h
i
n
e
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
l
a
n
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
l
a
n
t
_
i
d
"
)
,

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
m
a
c
h
i
n
e
_
n
a
m
e
"
)
)
.
a
l
i
a
s
(
"
m
a
c
h
i
n
e
_
n
a
m
e
"
)
,
 
 
k
e
e
p
 
'
I
M
M
-
2
2
'
 
a
s
-
i
s

 
F
.
c
o
l
(
"
t
o
n
n
a
g
e
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
t
o
n
n
a
g
e
"
)
,

 
s
t
d
_
n
a
m
e
(
F
.
c
o
l
(
"
m
a
n
u
f
a
c
t
u
r
e
r
"
)
)
.
a
l
i
a
s
(
"
m
a
n
u
f
a
c
t
u
r
e
r
"
)
,

 
F
.
c
o
l
(
"
i
n
s
t
a
l
l
_
y
e
a
r
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
i
n
s
t
a
l
l
_
y
e
a
r
"
)
,

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
s
t
a
t
u
s
"
)
)
.
a
l
i
a
s
(
"
s
t
a
t
u
s
"
)
,

 
F
.
c
o
l
(
"
_
b
a
t
c
h
_
i
d
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
s
y
s
t
e
m
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
f
i
l
e
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)
,
 
[
"
m
a
c
h
i
n
e
_
i
d
"
]
)

d
i
m
_
m
a
c
h
i
n
e
 
=
 
(
d
i
m
_
m
a
c
h
i
n
e
.
w
i
t
h
C
o
l
u
m
n
(
"
m
a
c
h
i
n
e
_
k
e
y
"
,
 
F
.
m
d
5
(
F
.
c
o
l
(
"
m
a
c
h
i
n
e
_
i
d
"
)
)
)

 
.
j
o
i
n
(
p
l
a
n
t
_
k
e
y
s
,
 
"
p
l
a
n
t
_
i
d
"
,
 
"
l
e
f
t
"
)

 
.
s
e
l
e
c
t
(
"
m
a
c
h
i
n
e
_
k
e
y
"
,
 
"
m
a
c
h
i
n
e
_
i
d
"
,
 
"
m
a
c
h
i
n
e
_
n
a
m
e
"
,
 
"
p
l
a
n
t
_
k
e
y
"
,

 
"
p
l
a
n
t
_
i
d
"
,
 
"
t
o
n
n
a
g
e
"
,
 
"
m
a
n
u
f
a
c
t
u
r
e
r
"
,
 
"
i
n
s
t
a
l
l
_
y
e
a
r
"
,

 
"
s
t
a
t
u
s
"
)
)


r
e
t
a
i
l
e
r
_
n
a
m
e
s
 
=
 
{
"
R
0
1
"
:
 
"
W
a
l
m
a
r
t
"
,
 
"
R
0
2
"
:
 
"
T
a
r
g
e
t
"
,
 
"
R
0
3
"
:
 
"
L
o
w
e
'
s
"
,

 
"
R
0
4
"
:
 
"
H
o
m
e
 
D
e
p
o
t
"
,
 
"
R
0
5
"
:
 
"
C
o
s
t
c
o
"
}

r
e
t
a
i
l
e
r
_
i
d
s
 
=
 
d
e
d
u
p
e
(

 
b
r
o
n
z
e
(
"
s
a
l
e
s
_
o
r
d
e
r
s
"
)
.
s
e
l
e
c
t
(
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
r
e
t
a
i
l
e
r
_
i
d
"
)
)
.
a
l
i
a
s
(
"
r
e
t
a
i
l
e
r
_
i
d
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)

 
.
u
n
i
o
n
B
y
N
a
m
e
(
b
r
o
n
z
e
(
"
r
e
t
a
i
l
_
p
o
s
"
)
.
s
e
l
e
c
t
(

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
r
e
t
a
i
l
e
r
_
i
d
"
)
)
.
a
l
i
a
s
(
"
r
e
t
a
i
l
e
r
_
i
d
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)
)
,
 
[
"
r
e
t
a
i
l
e
r
_
i
d
"
]
)

r
e
t
a
i
l
e
r
_
n
a
m
e
_
m
a
p
 
=
 
F
.
c
r
e
a
t
e
_
m
a
p
(

 
[
F
.
l
i
t
(
x
)
 
f
o
r
 
p
a
i
r
 
i
n
 
r
e
t
a
i
l
e
r
_
n
a
m
e
s
.
i
t
e
m
s
(
)
 
f
o
r
 
x
 
i
n
 
p
a
i
r
]
)

d
i
m
_
r
e
t
a
i
l
e
r
 
=
 
(
r
e
t
a
i
l
e
r
_
i
d
s

 
.
w
i
t
h
C
o
l
u
m
n
(
"
r
e
t
a
i
l
e
r
_
n
a
m
e
"
,

 
F
.
c
o
a
l
e
s
c
e
(
r
e
t
a
i
l
e
r
_
n
a
m
e
_
m
a
p
.
g
e
t
I
t
e
m
(
F
.
c
o
l
(
"
r
e
t
a
i
l
e
r
_
i
d
"
)
)
,

 
F
.
c
o
l
(
"
r
e
t
a
i
l
e
r
_
i
d
"
)
)
)

 
.
w
i
t
h
C
o
l
u
m
n
(
"
r
e
t
a
i
l
e
r
_
k
e
y
"
,
 
F
.
m
d
5
(
F
.
c
o
l
(
"
r
e
t
a
i
l
e
r
_
i
d
"
)
)
)

 
.
s
e
l
e
c
t
(
"
r
e
t
a
i
l
e
r
_
k
e
y
"
,
 
"
r
e
t
a
i
l
e
r
_
i
d
"
,
 
"
r
e
t
a
i
l
e
r
_
n
a
m
e
"
)
)


f
o
r
 
d
f
,
 
n
a
m
e
 
i
n
 
[
(
d
i
m
_
p
l
a
n
t
,
 
"
d
i
m
_
p
l
a
n
t
"
)
,
 
(
d
i
m
_
m
a
c
h
i
n
e
,
 
"
d
i
m
_
m
a
c
h
i
n
e
"
)
,

 
(
d
i
m
_
r
e
t
a
i
l
e
r
,
 
"
d
i
m
_
r
e
t
a
i
l
e
r
"
)
]
:

 
w
r
i
t
e
_
d
e
l
t
a
(
d
f
,
 
n
a
m
e
)

p
r
i
n
t
(
"
c
o
n
f
o
r
m
e
d
 
d
i
m
s
 
w
r
i
t
t
e
n
:
 
d
i
m
_
p
l
a
n
t
,
 
d
i
m
_
m
a
c
h
i
n
e
,
 
d
i
m
_
r
e
t
a
i
l
e
r
"
)


In [ ]:
T
A
R
G
E
T
 
=
 
f
"
{
T
A
B
L
E
S
}
/
d
i
m
_
p
r
o
d
u
c
t
"

p
r
o
d
_
l
o
o
k
u
p
 
=
 
d
e
d
u
p
e
(

 
b
r
o
n
z
e
(
"
p
r
o
d
u
c
t
s
"
)
.
s
e
l
e
c
t
(

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
r
o
d
u
c
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
r
o
d
u
c
t
_
i
d
"
)
,

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
s
k
u
"
)
)
.
a
l
i
a
s
(
"
s
k
u
"
)
,

 
s
t
d
_
n
a
m
e
(
F
.
c
o
l
(
"
p
r
o
d
u
c
t
_
n
a
m
e
"
)
)
.
a
l
i
a
s
(
"
p
r
o
d
u
c
t
_
n
a
m
e
"
)
,

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
c
a
t
e
g
o
r
y
"
)
)
.
a
l
i
a
s
(
"
c
a
t
e
g
o
r
y
"
)
,
 
 
c
l
e
a
n
 
i
n
 
t
h
e
 
E
R
P

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
r
e
s
i
n
_
t
y
p
e
"
)
)
.
a
l
i
a
s
(
"
r
e
s
i
n
_
t
y
p
e
"
)
,

 
F
.
c
o
l
(
"
u
n
i
t
_
w
e
i
g
h
t
_
k
g
"
)
.
c
a
s
t
(
"
d
e
c
i
m
a
l
(
6
,
2
)
"
)
.
a
l
i
a
s
(
"
u
n
i
t
_
w
e
i
g
h
t
_
k
g
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)
,
 
[
"
p
r
o
d
u
c
t
_
i
d
"
]
)


p
r
i
c
e
_
h
i
s
t
 
=
 
d
e
d
u
p
e
(

 
b
r
o
n
z
e
(
"
p
r
o
d
u
c
t
_
p
r
i
c
e
_
h
i
s
t
o
r
y
"
)
.
s
e
l
e
c
t
(

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
r
o
d
u
c
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
r
o
d
u
c
t
_
i
d
"
)
,

 
F
.
c
o
l
(
"
u
n
i
t
_
p
r
i
c
e
"
)
.
c
a
s
t
(
"
d
e
c
i
m
a
l
(
1
0
,
2
)
"
)
.
a
l
i
a
s
(
"
u
n
i
t
_
p
r
i
c
e
"
)
,

 
E
R
P
 
s
t
a
n
d
a
r
d
_
c
o
s
t
 
-
>
 
G
o
l
d
 
u
n
i
t
_
c
o
s
t
 
(
r
e
n
a
m
e
d
 
o
n
c
e
,
 
h
e
r
e
)

 
F
.
c
o
l
(
"
s
t
a
n
d
a
r
d
_
c
o
s
t
"
)
.
c
a
s
t
(
"
d
e
c
i
m
a
l
(
1
0
,
2
)
"
)
.
a
l
i
a
s
(
"
u
n
i
t
_
c
o
s
t
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
e
f
f
e
c
t
i
v
e
_
f
r
o
m
"
)
)
.
a
l
i
a
s
(
"
e
f
f
e
c
t
i
v
e
_
f
r
o
m
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
e
f
f
e
c
t
i
v
e
_
t
o
"
)
)
.
a
l
i
a
s
(
"
e
f
f
e
c
t
i
v
e
_
t
o
"
)
,

 
(
F
.
c
o
l
(
"
i
s
_
c
u
r
r
e
n
t
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
c
a
s
t
(
"
b
o
o
l
e
a
n
"
)
)
.
a
l
i
a
s
(
"
i
s
_
c
u
r
r
e
n
t
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)
,
 
[
"
p
r
o
d
u
c
t
_
i
d
"
,
 
"
e
f
f
e
c
t
i
v
e
_
f
r
o
m
"
]
)


i
n
c
o
m
i
n
g
 
=
 
(
p
r
i
c
e
_
h
i
s
t
.
j
o
i
n
(
p
r
o
d
_
l
o
o
k
u
p
.
d
r
o
p
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
,
 
"
p
r
o
d
u
c
t
_
i
d
"
,
 
"
l
e
f
t
"
)

 
.
w
i
t
h
C
o
l
u
m
n
(
"
p
r
o
d
u
c
t
_
k
e
y
"
,
 
F
.
m
d
5
(
F
.
c
o
n
c
a
t
_
w
s
(

 
"
|
"
,
 
"
p
r
o
d
u
c
t
_
i
d
"
,
 
F
.
d
a
t
e
_
f
o
r
m
a
t
(
"
e
f
f
e
c
t
i
v
e
_
f
r
o
m
"
,
 
"
y
y
y
y
-
M
M
-
d
d
"
)
)
)
)

 
.
s
e
l
e
c
t
(
"
p
r
o
d
u
c
t
_
k
e
y
"
,
 
"
p
r
o
d
u
c
t
_
i
d
"
,
 
"
s
k
u
"
,
 
"
p
r
o
d
u
c
t
_
n
a
m
e
"
,
 
"
c
a
t
e
g
o
r
y
"
,

 
"
r
e
s
i
n
_
t
y
p
e
"
,
 
"
u
n
i
t
_
w
e
i
g
h
t
_
k
g
"
,
 
"
u
n
i
t
_
p
r
i
c
e
"
,
 
"
u
n
i
t
_
c
o
s
t
"
,

 
"
e
f
f
e
c
t
i
v
e
_
f
r
o
m
"
,
 
"
e
f
f
e
c
t
i
v
e
_
t
o
"
,
 
"
i
s
_
c
u
r
r
e
n
t
"
)
)


i
f
 
n
o
t
 
D
e
l
t
a
T
a
b
l
e
.
i
s
D
e
l
t
a
T
a
b
l
e
(
s
p
a
r
k
,
 
T
A
R
G
E
T
)
:
 
 
f
i
r
s
t
 
l
o
a
d

 
i
n
c
o
m
i
n
g
.
w
r
i
t
e
.
f
o
r
m
a
t
(
"
d
e
l
t
a
"
)
.
m
o
d
e
(
"
o
v
e
r
w
r
i
t
e
"
)
.
s
a
v
e
(
T
A
R
G
E
T
)

e
l
s
e
:

 
t
g
t
 
=
 
D
e
l
t
a
T
a
b
l
e
.
f
o
r
P
a
t
h
(
s
p
a
r
k
,
 
T
A
R
G
E
T
)

 
c
h
a
n
g
e
d
 
=
 
i
n
c
o
m
i
n
g
.
f
i
l
t
e
r
(
"
i
s
_
c
u
r
r
e
n
t
"
)
.
a
l
i
a
s
(
"
s
"
)
.
j
o
i
n
(

 
t
g
t
.
t
o
D
F
(
)
.
f
i
l
t
e
r
(
"
i
s
_
c
u
r
r
e
n
t
"
)
.
a
l
i
a
s
(
"
t
"
)
,
 
"
p
r
o
d
u
c
t
_
i
d
"
)
.
f
i
l
t
e
r
(

 
"
s
.
u
n
i
t
_
p
r
i
c
e
 
<
>
 
t
.
u
n
i
t
_
p
r
i
c
e
 
O
R
 
s
.
u
n
i
t
_
c
o
s
t
 
<
>
 
t
.
u
n
i
t
_
c
o
s
t
 
"

 
"
O
R
 
s
.
p
r
o
d
u
c
t
_
n
a
m
e
 
<
>
 
t
.
p
r
o
d
u
c
t
_
n
a
m
e
 
O
R
 
s
.
c
a
t
e
g
o
r
y
 
<
>
 
t
.
c
a
t
e
g
o
r
y
"

 
)
.
s
e
l
e
c
t
(
"
s
.
p
r
o
d
u
c
t
_
i
d
"
)

 
1
)
 
e
x
p
i
r
e
 
t
h
e
 
c
u
r
r
e
n
t
 
r
o
w
 
f
o
r
 
a
n
y
t
h
i
n
g
 
t
h
a
t
 
c
h
a
n
g
e
d

 
(
t
g
t
.
a
l
i
a
s
(
"
t
"
)
.
m
e
r
g
e
(
c
h
a
n
g
e
d
.
a
l
i
a
s
(
"
s
"
)
,

 
"
t
.
p
r
o
d
u
c
t
_
i
d
 
=
 
s
.
p
r
o
d
u
c
t
_
i
d
 
A
N
D
 
t
.
i
s
_
c
u
r
r
e
n
t
"
)

 
.
w
h
e
n
M
a
t
c
h
e
d
U
p
d
a
t
e
(
s
e
t
=
{
"
i
s
_
c
u
r
r
e
n
t
"
:
 
"
f
a
l
s
e
"
,

 
"
e
f
f
e
c
t
i
v
e
_
t
o
"
:
 
"
c
u
r
r
e
n
t
_
d
a
t
e
(
)
"
}
)

 
.
e
x
e
c
u
t
e
(
)
)

 
2
)
 
i
n
s
e
r
t
 
t
h
e
 
n
e
w
 
v
e
r
s
i
o
n
 
o
f
 
c
h
a
n
g
e
d
 
p
r
o
d
u
c
t
s
 
+
 
3
)
 
b
r
a
n
d
-
n
e
w
 
p
r
o
d
u
c
t
s
:

 
e
v
e
r
y
 
i
n
c
o
m
i
n
g
 
v
e
r
s
i
o
n
 
w
h
o
s
e
 
p
r
o
d
u
c
t
_
k
e
y
 
i
s
 
n
o
t
 
y
e
t
 
i
n
 
t
h
e
 
t
a
r
g
e
t

 
t
g
t
 
=
 
D
e
l
t
a
T
a
b
l
e
.
f
o
r
P
a
t
h
(
s
p
a
r
k
,
 
T
A
R
G
E
T
)

 
n
e
w
_
r
o
w
s
 
=
 
i
n
c
o
m
i
n
g
.
j
o
i
n
(
t
g
t
.
t
o
D
F
(
)
,
 
[
"
p
r
o
d
u
c
t
_
k
e
y
"
]
,
 
"
l
e
f
t
_
a
n
t
i
"
)

 
(
t
g
t
.
a
l
i
a
s
(
"
t
"
)
.
m
e
r
g
e
(
n
e
w
_
r
o
w
s
.
a
l
i
a
s
(
"
s
"
)
,
 
"
1
 
=
 
0
"
)
 
 
p
u
r
e
-
i
n
s
e
r
t
 
m
e
r
g
e

 
.
w
h
e
n
N
o
t
M
a
t
c
h
e
d
I
n
s
e
r
t
(
v
a
l
u
e
s
=
{

 
"
p
r
o
d
u
c
t
_
k
e
y
"
:
 
"
s
.
p
r
o
d
u
c
t
_
k
e
y
"
,
 
"
p
r
o
d
u
c
t
_
i
d
"
:
 
"
s
.
p
r
o
d
u
c
t
_
i
d
"
,

 
"
s
k
u
"
:
 
"
s
.
s
k
u
"
,
 
"
p
r
o
d
u
c
t
_
n
a
m
e
"
:
 
"
s
.
p
r
o
d
u
c
t
_
n
a
m
e
"
,

 
"
c
a
t
e
g
o
r
y
"
:
 
"
s
.
c
a
t
e
g
o
r
y
"
,
 
"
r
e
s
i
n
_
t
y
p
e
"
:
 
"
s
.
r
e
s
i
n
_
t
y
p
e
"
,

 
"
u
n
i
t
_
w
e
i
g
h
t
_
k
g
"
:
 
"
s
.
u
n
i
t
_
w
e
i
g
h
t
_
k
g
"
,
 
"
u
n
i
t
_
p
r
i
c
e
"
:
 
"
s
.
u
n
i
t
_
p
r
i
c
e
"
,

 
"
u
n
i
t
_
c
o
s
t
"
:
 
"
s
.
u
n
i
t
_
c
o
s
t
"
,
 
"
e
f
f
e
c
t
i
v
e
_
f
r
o
m
"
:
 
"
s
.
e
f
f
e
c
t
i
v
e
_
f
r
o
m
"
,

 
"
e
f
f
e
c
t
i
v
e
_
t
o
"
:
 
"
s
.
e
f
f
e
c
t
i
v
e
_
t
o
"
,
 
"
i
s
_
c
u
r
r
e
n
t
"
:
 
"
s
.
i
s
_
c
u
r
r
e
n
t
"
}
)

 
.
e
x
e
c
u
t
e
(
)
)

p
r
i
n
t
(
"
d
i
m
_
p
r
o
d
u
c
t
:
 
S
C
D
 
T
y
p
e
 
2
 
m
e
r
g
e
 
c
o
m
p
l
e
t
e
"
)


In [ ]:
e
m
p
_
r
a
w
 
=
 
b
r
o
n
z
e
(
"
h
r
_
e
m
p
l
o
y
e
e
s
"
)

e
m
p
 
=
 
d
e
d
u
p
e
(
e
m
p
_
r
a
w
.
s
e
l
e
c
t
(

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
e
m
p
l
o
y
e
e
_
i
d
"
)
)
.
a
l
i
a
s
(
"
e
m
p
l
o
y
e
e
_
i
d
"
)
,

 
s
t
d
_
n
a
m
e
(
F
.
c
o
l
(
"
f
i
r
s
t
_
n
a
m
e
"
)
)
.
a
l
i
a
s
(
"
f
i
r
s
t
_
n
a
m
e
"
)
,

 
s
t
d
_
n
a
m
e
(
F
.
c
o
l
(
"
l
a
s
t
_
n
a
m
e
"
)
)
.
a
l
i
a
s
(
"
l
a
s
t
_
n
a
m
e
"
)
,

 
s
t
d
_
n
a
m
e
(
F
.
c
o
n
c
a
t
_
w
s
(
"
 
"
,
 
F
.
c
o
l
(
"
f
i
r
s
t
_
n
a
m
e
"
)
,
 
F
.
c
o
l
(
"
l
a
s
t
_
n
a
m
e
"
)
)

 
)
.
a
l
i
a
s
(
"
f
u
l
l
_
n
a
m
e
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
l
a
n
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
l
a
n
t
_
i
d
"
)
,

 
F
.
c
o
l
(
"
s
h
i
f
t
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
s
h
i
f
t
"
)
,

 
s
t
d
_
n
a
m
e
(
F
.
c
o
l
(
"
r
o
l
e
"
)
)
.
a
l
i
a
s
(
"
r
o
l
e
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
h
i
r
e
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
h
i
r
e
_
d
a
t
e
"
)
,

 
s
t
d
_
n
a
m
e
(
F
.
c
o
l
(
"
e
m
p
l
o
y
m
e
n
t
_
s
t
a
t
u
s
"
)
)
.
a
l
i
a
s
(
"
e
m
p
l
o
y
m
e
n
t
_
s
t
a
t
u
s
"
)
,

 
F
.
c
o
l
(
"
_
b
a
t
c
h
_
i
d
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
s
y
s
t
e
m
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
f
i
l
e
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)
,
 
[
"
e
m
p
l
o
y
e
e
_
i
d
"
]
)


e
m
p
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
e
m
p
,
 
F
.
c
o
l
(
"
h
i
r
e
_
d
a
t
e
"
)
.
i
s
N
o
t
N
u
l
l
(
)
,

 
"
e
m
p
l
o
y
e
e
:
 
u
n
p
a
r
s
e
a
b
l
e
 
o
r
 
m
i
s
s
i
n
g
 
h
i
r
e
_
d
a
t
e
"
)

e
m
p
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
e
m
p
,
 
F
.
c
o
l
(
"
p
l
a
n
t
_
i
d
"
)
.
i
s
i
n
(
[
r
.
p
l
a
n
t
_
i
d
 
f
o
r
 
r
 
i
n
 
d
i
m
_
p
l
a
n
t
.
c
o
l
l
e
c
t
(
)
]
)
,

 
"
e
m
p
l
o
y
e
e
:
 
p
l
a
n
t
_
i
d
 
n
o
t
 
i
n
 
D
i
m
P
l
a
n
t
"
)

e
m
p
 
=
 
(
e
m
p
.
w
i
t
h
C
o
l
u
m
n
(
"
e
m
p
l
o
y
e
e
_
k
e
y
"
,
 
F
.
m
d
5
(
F
.
c
o
l
(
"
e
m
p
l
o
y
e
e
_
i
d
"
)
)
)

 
.
j
o
i
n
(
p
l
a
n
t
_
k
e
y
s
,
 
"
p
l
a
n
t
_
i
d
"
,
 
"
l
e
f
t
"
)

 
.
s
e
l
e
c
t
(
"
e
m
p
l
o
y
e
e
_
k
e
y
"
,
 
"
e
m
p
l
o
y
e
e
_
i
d
"
,
 
"
f
u
l
l
_
n
a
m
e
"
,
 
"
f
i
r
s
t
_
n
a
m
e
"
,

 
"
l
a
s
t
_
n
a
m
e
"
,
 
"
p
l
a
n
t
_
k
e
y
"
,
 
"
p
l
a
n
t
_
i
d
"
,
 
"
s
h
i
f
t
"
,
 
"
r
o
l
e
"
,

 
"
h
i
r
e
_
d
a
t
e
"
,
 
"
e
m
p
l
o
y
m
e
n
t
_
s
t
a
t
u
s
"
)
)

w
r
i
t
e
_
d
e
l
t
a
(
e
m
p
,
 
"
d
i
m
_
e
m
p
l
o
y
e
e
"
)

p
r
i
n
t
(
f
"
d
i
m
_
e
m
p
l
o
y
e
e
 
r
o
w
s
:
 
{
e
m
p
.
c
o
u
n
t
(
)
}
"
)


In [ ]:
p
m
_
r
a
w
 
=
 
b
r
o
n
z
e
(
"
p
r
o
d
u
c
t
_
m
a
s
t
e
r
"
)

c
a
t
e
g
o
r
y
_
m
a
p
 
=
 
(
p
r
o
d
_
l
o
o
k
u
p
.
s
e
l
e
c
t
(

 
F
.
l
o
w
e
r
(
F
.
t
r
i
m
(
F
.
c
o
l
(
"
c
a
t
e
g
o
r
y
"
)
)
)
.
a
l
i
a
s
(
"
_
c
a
t
_
k
e
y
"
)
,

 
F
.
c
o
l
(
"
c
a
t
e
g
o
r
y
"
)
.
a
l
i
a
s
(
"
c
a
t
e
g
o
r
y
"
)
)
.
d
i
s
t
i
n
c
t
(
)
)

p
m
 
=
 
(
p
m
_
r
a
w
.
s
e
l
e
c
t
(

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
P
r
o
d
u
c
t
 
I
D
"
)
)
.
a
l
i
a
s
(
"
p
r
o
d
u
c
t
_
i
d
"
)
,

 
s
t
d
_
n
a
m
e
(
F
.
c
o
l
(
"
P
r
o
d
u
c
t
 
N
a
m
e
"
)
)
.
a
l
i
a
s
(
"
p
r
o
d
u
c
t
_
n
a
m
e
"
)
,

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
C
a
t
e
g
o
r
y
"
)
)
.
a
l
i
a
s
(
"
c
a
t
e
g
o
r
y
_
a
s
_
r
e
c
e
i
v
e
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
R
e
s
i
n
"
)
)
.
a
l
i
a
s
(
"
r
e
s
i
n
_
t
y
p
e
"
)
,

 
F
.
c
o
l
(
"
L
i
s
t
 
P
r
i
c
e
"
)
.
c
a
s
t
(
"
d
e
c
i
m
a
l
(
1
0
,
2
)
"
)
.
a
l
i
a
s
(
"
l
i
s
t
_
p
r
i
c
e
"
)
,

 
F
.
c
o
l
(
"
_
b
a
t
c
h
_
i
d
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
s
y
s
t
e
m
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
f
i
l
e
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)

 
.
j
o
i
n
(
c
a
t
e
g
o
r
y
_
m
a
p
,

 
F
.
l
o
w
e
r
(
F
.
c
o
l
(
"
c
a
t
e
g
o
r
y
_
a
s
_
r
e
c
e
i
v
e
d
"
)
)
 
=
=
 
F
.
c
o
l
(
"
_
c
a
t
_
k
e
y
"
)
,
 
"
l
e
f
t
"
)

 
.
d
r
o
p
(
"
_
c
a
t
_
k
e
y
"
)
)

p
m
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
p
m
,
 
F
.
c
o
l
(
"
c
a
t
e
g
o
r
y
"
)
.
i
s
N
o
t
N
u
l
l
(
)
,

 
"
D
Q
-
0
7
/
D
3
:
 
p
r
o
d
u
c
t
_
m
a
s
t
e
r
 
c
a
t
e
g
o
r
y
 
d
i
d
 
n
o
t
 
m
a
p
 
t
o
 
a
 
c
o
n
f
o
r
m
e
d
 
E
R
P
 
c
a
t
e
g
o
r
y
"
)

d
3
_
c
h
a
n
g
e
d
 
=
 
p
m
.
f
i
l
t
e
r
(
F
.
c
o
l
(
"
c
a
t
e
g
o
r
y
_
a
s
_
r
e
c
e
i
v
e
d
"
)
 
!
=
 
F
.
c
o
l
(
"
c
a
t
e
g
o
r
y
"
)
)
.
c
o
u
n
t
(
)

w
r
i
t
e
_
d
e
l
t
a
(
p
m
.
s
e
l
e
c
t
(
"
p
r
o
d
u
c
t
_
i
d
"
,
 
"
p
r
o
d
u
c
t
_
n
a
m
e
"
,
 
"
c
a
t
e
g
o
r
y
"
,

 
"
c
a
t
e
g
o
r
y
_
a
s
_
r
e
c
e
i
v
e
d
"
,
 
"
r
e
s
i
n
_
t
y
p
e
"
,
 
"
l
i
s
t
_
p
r
i
c
e
"
)
,

 
"
s
i
l
v
e
r
_
p
r
o
d
u
c
t
_
m
a
s
t
e
r
"
)

p
r
i
n
t
(
f
"
s
i
l
v
e
r
_
p
r
o
d
u
c
t
_
m
a
s
t
e
r
:
 
{
d
3
_
c
h
a
n
g
e
d
}
 
c
a
t
e
g
o
r
i
e
s
 
c
o
n
f
o
r
m
e
d
 
(
D
3
)
"
)


t
a
r
g
e
t
s
 
=
 
b
r
o
n
z
e
(
"
p
l
a
n
t
_
t
a
r
g
e
t
s
"
)
.
s
e
l
e
c
t
(

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
P
l
a
n
t
"
)
)
.
a
l
i
a
s
(
"
p
l
a
n
t
"
)
,

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
M
o
n
t
h
"
)
)
.
a
l
i
a
s
(
"
m
o
n
t
h
"
)
,

 
F
.
r
e
g
e
x
p
_
r
e
p
l
a
c
e
(
F
.
c
o
l
(
"
T
a
r
g
e
t
 
G
o
o
d
 
Q
t
y
"
)
,
 
"
,
"
,
 
"
"
)
.
c
a
s
t
(
"
i
n
t
"
)

 
.
a
l
i
a
s
(
"
t
a
r
g
e
t
_
g
o
o
d
_
q
t
y
"
)
,

 
F
.
c
o
l
(
"
T
a
r
g
e
t
 
S
c
r
a
p
 
%
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
a
l
i
a
s
(
"
t
a
r
g
e
t
_
s
c
r
a
p
_
p
c
t
"
)
,

 
F
.
c
o
l
(
"
T
a
r
g
e
t
 
O
T
I
F
 
%
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
a
l
i
a
s
(
"
t
a
r
g
e
t
_
o
t
i
f
_
p
c
t
"
)
)

w
r
i
t
e
_
d
e
l
t
a
(
t
a
r
g
e
t
s
,
 
"
s
i
l
v
e
r
_
p
l
a
n
t
_
t
a
r
g
e
t
s
"
)

p
r
i
n
t
(
"
s
i
l
v
e
r
_
p
l
a
n
t
_
t
a
r
g
e
t
s
 
w
r
i
t
t
e
n
 
(
F
i
n
a
n
c
e
 
t
e
x
t
 
q
u
a
n
t
i
t
i
e
s
 
p
a
r
s
e
d
 
t
o
 
i
n
t
)
"
)


In [ ]:
d
i
m
_
p
r
o
d
u
c
t
_
c
u
r
r
e
n
t
 
=
 
s
p
a
r
k
.
r
e
a
d
.
f
o
r
m
a
t
(
"
d
e
l
t
a
"
)
.
l
o
a
d
(
T
A
R
G
E
T
)
.
f
i
l
t
e
r
(
"
i
s
_
c
u
r
r
e
n
t
"
)

p
l
a
n
t
_
i
d
s
 
 
=
 
[
r
.
p
l
a
n
t
_
i
d
 
f
o
r
 
r
 
i
n
 
d
i
m
_
p
l
a
n
t
.
c
o
l
l
e
c
t
(
)
]

p
r
o
d
u
c
t
_
i
d
s
 
 
=
 
[
r
.
p
r
o
d
u
c
t
_
i
d
 
f
o
r
 
r
 
i
n
 
d
i
m
_
p
r
o
d
u
c
t
_
c
u
r
r
e
n
t
.
c
o
l
l
e
c
t
(
)
]

e
m
p
l
o
y
e
e
_
i
d
s
 
=
 
[
r
.
e
m
p
l
o
y
e
e
_
i
d
 
f
o
r
 
r
 
i
n
 
e
m
p
.
c
o
l
l
e
c
t
(
)
]


p
r
o
d
 
=
 
d
e
d
u
p
e
(
b
r
o
n
z
e
(
"
p
r
o
d
u
c
t
i
o
n
_
r
u
n
s
"
)
.
s
e
l
e
c
t
(

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
r
u
n
_
i
d
"
)
)
.
a
l
i
a
s
(
"
r
u
n
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
w
o
r
k
_
o
r
d
e
r
_
i
d
"
)
)
.
a
l
i
a
s
(
"
w
o
r
k
_
o
r
d
e
r
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
m
a
c
h
i
n
e
_
i
d
"
)
)
.
a
l
i
a
s
(
"
m
a
c
h
i
n
e
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
l
a
n
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
l
a
n
t
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
r
o
d
u
c
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
r
o
d
u
c
t
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
o
p
e
r
a
t
o
r
_
i
d
"
)
)
.
a
l
i
a
s
(
"
o
p
e
r
a
t
o
r
_
i
d
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
r
u
n
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
r
u
n
_
d
a
t
e
"
)
,

 
F
.
c
o
l
(
"
s
h
i
f
t
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
s
h
i
f
t
"
)
,

 
F
.
c
o
l
(
"
p
l
a
n
n
e
d
_
q
t
y
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
p
l
a
n
n
e
d
_
q
t
y
"
)
,

 
F
.
c
o
l
(
"
g
o
o
d
_
q
t
y
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
g
o
o
d
_
q
t
y
"
)
,

 
F
.
c
o
l
(
"
s
c
r
a
p
_
q
t
y
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
a
l
i
a
s
(
"
s
c
r
a
p
_
q
t
y
"
)
,
 
 
c
a
s
t
 
t
o
 
i
n
t
 
A
F
T
E
R
 
t
h
e
 
D
2
 
n
u
l
l
 
c
h
e
c
k

 
F
.
c
o
l
(
"
d
o
w
n
t
i
m
e
_
m
i
n
u
t
e
s
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
d
o
w
n
t
i
m
e
_
m
i
n
u
t
e
s
"
)
,

 
F
.
c
o
l
(
"
r
e
s
i
n
_
u
s
e
d
_
k
g
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
a
l
i
a
s
(
"
r
e
s
i
n
_
u
s
e
d
_
k
g
"
)
,

 
F
.
c
o
l
(
"
c
y
c
l
e
_
t
i
m
e
_
s
e
c
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
a
l
i
a
s
(
"
c
y
c
l
e
_
t
i
m
e
_
s
e
c
"
)
,

 
F
.
c
o
l
(
"
_
b
a
t
c
h
_
i
d
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
s
y
s
t
e
m
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
f
i
l
e
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)
,
 
[
"
r
u
n
_
i
d
"
]
)
 
 
D
1
:
 
d
e
d
u
p
e
 
o
n
 
r
u
n
_
i
d

p
r
o
d
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
p
r
o
d
,
 
F
.
c
o
l
(
"
s
c
r
a
p
_
q
t
y
"
)
.
i
s
N
o
t
N
u
l
l
(
)
,

 
"
D
Q
-
0
2
/
D
2
:
 
n
u
l
l
 
s
c
r
a
p
_
q
t
y
"
)
 
 
D
2
 
-
>
 
q
u
a
r
a
n
t
i
n
e

p
r
o
d
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
p
r
o
d
,
 
(
F
.
c
o
l
(
"
g
o
o
d
_
q
t
y
"
)
 
>
=
 
0
)
 
&
 
(
F
.
c
o
l
(
"
s
c
r
a
p
_
q
t
y
"
)
 
>
=
 
0
)
,

 
"
D
Q
-
0
5
/
D
6
:
 
n
e
g
a
t
i
v
e
 
q
u
a
n
t
i
t
y
"
)
 
 
D
6
 
-
>
 
q
u
a
r
a
n
t
i
n
e

p
r
o
d
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
p
r
o
d
,
 
F
.
c
o
l
(
"
o
p
e
r
a
t
o
r
_
i
d
"
)
.
i
s
i
n
(
e
m
p
l
o
y
e
e
_
i
d
s
)
,

 
"
D
Q
-
0
3
/
D
4
:
 
o
r
p
h
a
n
 
o
p
e
r
a
t
o
r
_
i
d
 
n
o
t
 
i
n
 
H
R
 
e
m
p
l
o
y
e
e
s
"
)

p
r
o
d
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
p
r
o
d
,
 
F
.
c
o
l
(
"
p
r
o
d
u
c
t
_
i
d
"
)
.
i
s
i
n
(
p
r
o
d
u
c
t
_
i
d
s
)
,

 
"
D
Q
-
0
3
:
 
o
r
p
h
a
n
 
p
r
o
d
u
c
t
_
i
d
 
n
o
t
 
i
n
 
D
i
m
P
r
o
d
u
c
t
"
)

p
r
o
d
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
p
r
o
d
,
 
F
.
c
o
l
(
"
p
l
a
n
t
_
i
d
"
)
.
i
s
i
n
(
p
l
a
n
t
_
i
d
s
)
,

 
"
p
r
o
d
u
c
t
i
o
n
_
r
u
n
:
 
p
l
a
n
t
_
i
d
 
n
o
t
 
i
n
 
D
i
m
P
l
a
n
t
 
(
o
r
p
h
a
n
 
F
K
)
"
)

p
r
o
d
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
p
r
o
d
,
 
F
.
c
o
l
(
"
r
u
n
_
d
a
t
e
"
)
.
i
s
N
o
t
N
u
l
l
(
)
,

 
"
D
Q
-
0
6
:
 
u
n
p
a
r
s
e
a
b
l
e
 
r
u
n
_
d
a
t
e
"
)

p
r
o
d
 
=
 
p
r
o
d
.
w
i
t
h
C
o
l
u
m
n
(
"
s
c
r
a
p
_
q
t
y
"
,
 
F
.
c
o
l
(
"
s
c
r
a
p
_
q
t
y
"
)
.
c
a
s
t
(
"
i
n
t
"
)
)

w
r
i
t
e
_
d
e
l
t
a
(
p
r
o
d
,
 
"
s
i
l
v
e
r
_
p
r
o
d
u
c
t
i
o
n
_
r
u
n
s
"
)


w
o
r
k
_
o
r
d
e
r
s
 
=
 
d
e
d
u
p
e
(
b
r
o
n
z
e
(
"
w
o
r
k
_
o
r
d
e
r
s
"
)
.
s
e
l
e
c
t
(

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
w
o
r
k
_
o
r
d
e
r
_
i
d
"
)
)
.
a
l
i
a
s
(
"
w
o
r
k
_
o
r
d
e
r
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
r
o
d
u
c
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
r
o
d
u
c
t
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
l
a
n
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
l
a
n
t
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
m
a
c
h
i
n
e
_
i
d
"
)
)
.
a
l
i
a
s
(
"
m
a
c
h
i
n
e
_
i
d
"
)
,

 
F
.
c
o
l
(
"
p
l
a
n
n
e
d
_
q
t
y
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
p
l
a
n
n
e
d
_
q
t
y
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
s
t
a
r
t
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
s
t
a
r
t
_
d
a
t
e
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
d
u
e
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
d
u
e
_
d
a
t
e
"
)
,

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
s
t
a
t
u
s
"
)
)
.
a
l
i
a
s
(
"
s
t
a
t
u
s
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
c
r
e
a
t
e
d
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
c
r
e
a
t
e
d
_
d
a
t
e
"
)
,

 
F
.
c
o
l
(
"
_
b
a
t
c
h
_
i
d
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
s
y
s
t
e
m
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
f
i
l
e
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)
,
 
[
"
w
o
r
k
_
o
r
d
e
r
_
i
d
"
]
)

w
r
i
t
e
_
d
e
l
t
a
(
w
o
r
k
_
o
r
d
e
r
s
,
 
"
s
i
l
v
e
r
_
w
o
r
k
_
o
r
d
e
r
s
"
)


q
i
 
=
 
d
e
d
u
p
e
(
b
r
o
n
z
e
(
"
q
u
a
l
i
t
y
_
i
n
s
p
e
c
t
i
o
n
s
"
)
.
s
e
l
e
c
t
(

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
i
n
s
p
e
c
t
i
o
n
_
i
d
"
)
)
.
a
l
i
a
s
(
"
i
n
s
p
e
c
t
i
o
n
_
i
d
"
)
,

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
r
u
n
_
i
d
"
)
)
.
a
l
i
a
s
(
"
r
u
n
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
r
o
d
u
c
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
r
o
d
u
c
t
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
l
a
n
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
l
a
n
t
_
i
d
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
i
n
s
p
e
c
t
i
o
n
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
i
n
s
p
e
c
t
i
o
n
_
d
a
t
e
"
)
,

 
F
.
c
o
l
(
"
i
n
s
p
e
c
t
e
d
_
q
t
y
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
i
n
s
p
e
c
t
e
d
_
q
t
y
"
)
,

 
F
.
c
o
l
(
"
d
e
f
e
c
t
_
q
t
y
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
d
e
f
e
c
t
_
q
t
y
"
)
,

 
s
t
d
_
n
a
m
e
(
F
.
c
o
l
(
"
d
e
f
e
c
t
_
t
y
p
e
"
)
)
.
a
l
i
a
s
(
"
d
e
f
e
c
t
_
t
y
p
e
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
i
n
s
p
e
c
t
o
r
_
i
d
"
)
)
.
a
l
i
a
s
(
"
i
n
s
p
e
c
t
o
r
_
i
d
"
)
,

 
s
t
d
_
n
a
m
e
(
F
.
c
o
l
(
"
r
e
s
u
l
t
"
)
)
.
a
l
i
a
s
(
"
r
e
s
u
l
t
"
)
,

 
F
.
c
o
l
(
"
_
b
a
t
c
h
_
i
d
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
s
y
s
t
e
m
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
f
i
l
e
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)
,
 
[
"
i
n
s
p
e
c
t
i
o
n
_
i
d
"
]
)

q
i
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
q
i
,
 
F
.
c
o
l
(
"
p
r
o
d
u
c
t
_
i
d
"
)
.
i
s
i
n
(
p
r
o
d
u
c
t
_
i
d
s
)
,

 
"
q
u
a
l
i
t
y
_
i
n
s
p
e
c
t
i
o
n
:
 
p
r
o
d
u
c
t
_
i
d
 
n
o
t
 
i
n
 
D
i
m
P
r
o
d
u
c
t
 
(
o
r
p
h
a
n
 
F
K
)
"
)

q
i
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
q
i
,
 
F
.
c
o
l
(
"
d
e
f
e
c
t
_
q
t
y
"
)
 
>
=
 
0
,

 
"
q
u
a
l
i
t
y
_
i
n
s
p
e
c
t
i
o
n
:
 
n
e
g
a
t
i
v
e
 
d
e
f
e
c
t
_
q
t
y
"
)

w
r
i
t
e
_
d
e
l
t
a
(
q
i
,
 
"
s
i
l
v
e
r
_
q
u
a
l
i
t
y
_
i
n
s
p
e
c
t
i
o
n
s
"
)


o
r
d
e
r
s
 
=
 
d
e
d
u
p
e
(
b
r
o
n
z
e
(
"
s
a
l
e
s
_
o
r
d
e
r
s
"
)
.
s
e
l
e
c
t
(

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
s
a
l
e
s
_
o
r
d
e
r
_
i
d
"
)
)
.
a
l
i
a
s
(
"
s
a
l
e
s
_
o
r
d
e
r
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
r
e
t
a
i
l
e
r
_
i
d
"
)
)
.
a
l
i
a
s
(
"
r
e
t
a
i
l
e
r
_
i
d
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
o
r
d
e
r
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
o
r
d
e
r
_
d
a
t
e
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
p
r
o
m
i
s
e
d
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
p
r
o
m
i
s
e
d
_
d
a
t
e
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
f
u
l
f
i
l
l
i
n
g
_
p
l
a
n
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
f
u
l
f
i
l
l
i
n
g
_
p
l
a
n
t
_
i
d
"
)
,

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
s
t
a
t
u
s
"
)
)
.
a
l
i
a
s
(
"
s
t
a
t
u
s
"
)
,

 
F
.
c
o
l
(
"
_
b
a
t
c
h
_
i
d
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
s
y
s
t
e
m
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
f
i
l
e
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)
,
 
[
"
s
a
l
e
s
_
o
r
d
e
r
_
i
d
"
]
)

w
r
i
t
e
_
d
e
l
t
a
(
o
r
d
e
r
s
,
 
"
s
i
l
v
e
r
_
s
a
l
e
s
_
o
r
d
e
r
s
"
)


l
i
n
e
s
 
=
 
d
e
d
u
p
e
(
b
r
o
n
z
e
(
"
s
a
l
e
s
_
o
r
d
e
r
_
l
i
n
e
s
"
)
.
s
e
l
e
c
t
(

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
s
a
l
e
s
_
o
r
d
e
r
_
l
i
n
e
_
i
d
"
)
)
.
a
l
i
a
s
(
"
s
a
l
e
s
_
o
r
d
e
r
_
l
i
n
e
_
i
d
"
)
,

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
s
a
l
e
s
_
o
r
d
e
r
_
i
d
"
)
)
.
a
l
i
a
s
(
"
s
a
l
e
s
_
o
r
d
e
r
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
r
o
d
u
c
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
r
o
d
u
c
t
_
i
d
"
)
,

 
F
.
c
o
l
(
"
o
r
d
e
r
e
d
_
q
t
y
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
o
r
d
e
r
e
d
_
q
t
y
"
)
,

 
F
.
c
o
l
(
"
u
n
i
t
_
p
r
i
c
e
"
)
.
c
a
s
t
(
"
d
e
c
i
m
a
l
(
1
0
,
2
)
"
)
.
a
l
i
a
s
(
"
u
n
i
t
_
p
r
i
c
e
"
)
,

 
F
.
c
o
l
(
"
_
b
a
t
c
h
_
i
d
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
s
y
s
t
e
m
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
f
i
l
e
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)
,
 
[
"
s
a
l
e
s
_
o
r
d
e
r
_
l
i
n
e
_
i
d
"
]
)

l
i
n
e
s
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
l
i
n
e
s
,
 
F
.
c
o
l
(
"
o
r
d
e
r
e
d
_
q
t
y
"
)
 
>
 
0
,

 
"
s
a
l
e
s
_
o
r
d
e
r
_
l
i
n
e
:
 
o
r
d
e
r
e
d
_
q
t
y
 
<
=
 
0
"
)

l
i
n
e
s
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
l
i
n
e
s
,
 
F
.
c
o
l
(
"
p
r
o
d
u
c
t
_
i
d
"
)
.
i
s
i
n
(
p
r
o
d
u
c
t
_
i
d
s
)
,

 
"
s
a
l
e
s
_
o
r
d
e
r
_
l
i
n
e
:
 
p
r
o
d
u
c
t
_
i
d
 
n
o
t
 
i
n
 
D
i
m
P
r
o
d
u
c
t
 
(
o
r
p
h
a
n
 
F
K
)
"
)

w
r
i
t
e
_
d
e
l
t
a
(
l
i
n
e
s
,
 
"
s
i
l
v
e
r
_
s
a
l
e
s
_
o
r
d
e
r
_
l
i
n
e
s
"
)


s
h
i
p
 
=
 
d
e
d
u
p
e
(
b
r
o
n
z
e
(
"
s
h
i
p
m
e
n
t
s
"
)
.
s
e
l
e
c
t
(

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
s
h
i
p
m
e
n
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
s
h
i
p
m
e
n
t
_
i
d
"
)
,

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
s
a
l
e
s
_
o
r
d
e
r
_
l
i
n
e
_
i
d
"
)
)
.
a
l
i
a
s
(
"
s
a
l
e
s
_
o
r
d
e
r
_
l
i
n
e
_
i
d
"
)
,

 
F
.
t
r
i
m
(
F
.
c
o
l
(
"
s
a
l
e
s
_
o
r
d
e
r
_
i
d
"
)
)
.
a
l
i
a
s
(
"
s
a
l
e
s
_
o
r
d
e
r
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
l
a
n
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
l
a
n
t
_
i
d
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
s
h
i
p
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
s
h
i
p
_
d
a
t
e
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
d
e
l
i
v
e
r
y
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
d
e
l
i
v
e
r
y
_
d
a
t
e
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
p
r
o
m
i
s
e
d
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
p
r
o
m
i
s
e
d
_
d
a
t
e
"
)
,

 
F
.
c
o
l
(
"
s
h
i
p
p
e
d
_
q
t
y
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
s
h
i
p
p
e
d
_
q
t
y
"
)
,

 
s
t
d
_
n
a
m
e
(
F
.
c
o
l
(
"
c
a
r
r
i
e
r
"
)
)
.
a
l
i
a
s
(
"
c
a
r
r
i
e
r
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
r
e
c
o
r
d
_
c
r
e
a
t
e
d
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
r
e
c
o
r
d
_
c
r
e
a
t
e
d
_
d
a
t
e
"
)
,

 
F
.
c
o
l
(
"
_
b
a
t
c
h
_
i
d
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
s
y
s
t
e
m
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
f
i
l
e
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)
,
 
[
"
s
h
i
p
m
e
n
t
_
i
d
"
]
)

s
h
i
p
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
s
h
i
p
,
 
F
.
c
o
l
(
"
s
h
i
p
p
e
d
_
q
t
y
"
)
 
>
=
 
0
,
 
"
s
h
i
p
m
e
n
t
:
 
n
e
g
a
t
i
v
e
 
s
h
i
p
p
e
d
_
q
t
y
"
)

w
r
i
t
e
_
d
e
l
t
a
(
s
h
i
p
,
 
"
s
i
l
v
e
r
_
s
h
i
p
m
e
n
t
s
"
)


i
n
v
 
=
 
d
e
d
u
p
e
(
b
r
o
n
z
e
(
"
i
n
v
e
n
t
o
r
y
_
d
a
i
l
y
"
)
.
s
e
l
e
c
t
(

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
i
n
v
e
n
t
o
r
y
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
i
n
v
e
n
t
o
r
y
_
d
a
t
e
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
l
a
n
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
l
a
n
t
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
r
e
s
i
n
_
t
y
p
e
"
)
)
.
a
l
i
a
s
(
"
r
e
s
i
n
_
t
y
p
e
"
)
,

 
F
.
c
o
l
(
"
o
n
_
h
a
n
d
_
k
g
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
a
l
i
a
s
(
"
o
n
_
h
a
n
d
_
k
g
"
)
,

 
F
.
c
o
l
(
"
c
o
n
s
u
m
e
d
_
k
g
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
a
l
i
a
s
(
"
c
o
n
s
u
m
e
d
_
k
g
"
)
,

 
F
.
c
o
l
(
"
r
e
c
e
i
v
e
d
_
k
g
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
a
l
i
a
s
(
"
r
e
c
e
i
v
e
d
_
k
g
"
)
,

 
F
.
c
o
l
(
"
s
a
f
e
t
y
_
s
t
o
c
k
_
k
g
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
a
l
i
a
s
(
"
s
a
f
e
t
y
_
s
t
o
c
k
_
k
g
"
)
,

 
F
.
c
o
l
(
"
_
b
a
t
c
h
_
i
d
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
s
y
s
t
e
m
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
f
i
l
e
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)
,
 
[
"
i
n
v
e
n
t
o
r
y
_
d
a
t
e
"
,
 
"
p
l
a
n
t
_
i
d
"
,
 
"
r
e
s
i
n
_
t
y
p
e
"
]
)

w
r
i
t
e
_
d
e
l
t
a
(
i
n
v
,
 
"
s
i
l
v
e
r
_
i
n
v
e
n
t
o
r
y
_
d
a
i
l
y
"
)


p
o
s
 
=
 
b
r
o
n
z
e
(
"
r
e
t
a
i
l
_
p
o
s
"
)
.
s
e
l
e
c
t
(

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
r
e
t
a
i
l
e
r
_
i
d
"
)
)
.
a
l
i
a
s
(
"
r
e
t
a
i
l
e
r
_
i
d
"
)
,

 
s
t
d
_
c
o
d
e
(
F
.
c
o
l
(
"
p
r
o
d
u
c
t
_
i
d
"
)
)
.
a
l
i
a
s
(
"
p
r
o
d
u
c
t
_
i
d
"
)
,

 
p
a
r
s
e
_
m
i
x
e
d
_
d
a
t
e
(
F
.
c
o
l
(
"
s
a
l
e
_
d
a
t
e
"
)
)
.
a
l
i
a
s
(
"
s
a
l
e
_
d
a
t
e
"
)
,
 
 
D
7
:
 
m
i
x
e
d
 
f
o
r
m
a
t
s

 
F
.
c
o
l
(
"
s
o
l
d
_
q
t
y
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
a
l
i
a
s
(
"
s
o
l
d
_
q
t
y
"
)
,

 
F
.
c
o
l
(
"
r
e
t
u
r
n
s
_
q
t
y
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
r
e
t
u
r
n
s
_
q
t
y
"
)
,

 
F
.
c
o
l
(
"
p
r
o
m
o
_
f
l
a
g
"
)
.
c
a
s
t
(
"
d
o
u
b
l
e
"
)
.
c
a
s
t
(
"
i
n
t
"
)
.
a
l
i
a
s
(
"
p
r
o
m
o
_
f
l
a
g
"
)
,

 
F
.
c
o
l
(
"
_
b
a
t
c
h
_
i
d
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
s
y
s
t
e
m
"
)
,
 
F
.
c
o
l
(
"
_
s
o
u
r
c
e
_
f
i
l
e
"
)
,

 
F
.
c
o
l
(
"
_
i
n
g
e
s
t
e
d
_
a
t
"
)
)

p
o
s
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
p
o
s
,
 
F
.
c
o
l
(
"
s
a
l
e
_
d
a
t
e
"
)
.
i
s
N
o
t
N
u
l
l
(
)
,

 
"
D
Q
-
0
6
/
D
7
:
 
u
n
p
a
r
s
e
a
b
l
e
 
s
a
l
e
_
d
a
t
e
"
)

p
o
s
 
=
 
s
p
l
i
t
_
v
a
l
i
d
(
p
o
s
,
 
F
.
c
o
l
(
"
s
o
l
d
_
q
t
y
"
)
 
>
=
 
0
,

 
"
D
Q
-
0
5
/
D
6
:
 
n
e
g
a
t
i
v
e
 
s
o
l
d
_
q
t
y
"
)

p
o
s
 
=
 
p
o
s
.
w
i
t
h
C
o
l
u
m
n
(
"
s
o
l
d
_
q
t
y
"
,
 
F
.
c
o
l
(
"
s
o
l
d
_
q
t
y
"
)
.
c
a
s
t
(
"
i
n
t
"
)
)

w
r
i
t
e
_
d
e
l
t
a
(
p
o
s
,
 
"
s
i
l
v
e
r
_
r
e
t
a
i
l
_
p
o
s
"
)

p
r
i
n
t
(
"
s
i
l
v
e
r
 
t
r
a
n
s
a
c
t
i
o
n
a
l
 
t
a
b
l
e
s
 
w
r
i
t
t
e
n
"
)


In [ ]:
 
S
h
i
p
m
e
n
t
s
 
a
r
r
i
v
e
 
a
t
 
s
h
i
p
m
e
n
t
 
g
r
a
i
n
 
—
 
p
o
t
e
n
t
i
a
l
l
y
 
m
a
n
y
 
p
e
r
 
o
r
d
e
r
 
l
i
n
e
.

 
P
r
e
-
a
g
g
r
e
g
a
t
e
 
t
o
 
t
h
e
 
O
R
D
E
R
-
L
I
N
E
 
g
r
a
i
n
 
(
s
a
l
e
s
_
o
r
d
e
r
_
l
i
n
e
_
i
d
)
 
b
e
f
o
r
e
 
a
n
y
 
j
o
i
n
:

s
h
i
p
_
b
y
_
l
i
n
e
 
=
 
(
s
h
i
p
.
g
r
o
u
p
B
y
(
"
s
a
l
e
s
_
o
r
d
e
r
_
l
i
n
e
_
i
d
"
,
 
"
s
a
l
e
s
_
o
r
d
e
r
_
i
d
"
)

 
.
a
g
g
(
F
.
s
u
m
(
"
s
h
i
p
p
e
d
_
q
t
y
"
)
.
a
l
i
a
s
(
"
s
h
i
p
p
e
d
_
q
t
y
"
)
,

 
F
.
m
a
x
(
"
s
h
i
p
_
d
a
t
e
"
)
.
a
l
i
a
s
(
"
s
h
i
p
_
d
a
t
e
"
)
,

 
F
.
m
a
x
(
"
d
e
l
i
v
e
r
y
_
d
a
t
e
"
)
.
a
l
i
a
s
(
"
d
e
l
i
v
e
r
y
_
d
a
t
e
"
)
,

 
F
.
f
i
r
s
t
(
"
p
l
a
n
t
_
i
d
"
)
.
a
l
i
a
s
(
"
s
h
i
p
m
e
n
t
_
p
l
a
n
t
_
i
d
"
)
,

 
F
.
c
o
u
n
t
(
"
s
h
i
p
m
e
n
t
_
i
d
"
)
.
a
l
i
a
s
(
"
s
h
i
p
m
e
n
t
_
c
o
u
n
t
"
)
)
)


E
n
r
i
c
h
 
l
i
n
e
s
 
w
i
t
h
 
t
h
e
 
o
r
d
e
r
 
h
e
a
d
e
r
 
(
r
e
t
a
i
l
e
r
,
 
d
a
t
e
s
,
 
f
u
l
f
i
l
l
i
n
g
 
p
l
a
n
t
)
 
a
t

 
h
e
a
d
e
r
 
g
r
a
i
n
 
—
 
o
n
e
 
h
e
a
d
e
r
 
r
o
w
 
p
e
r
 
s
a
l
e
s
_
o
r
d
e
r
_
i
d
,
 
s
o
 
t
h
i
s
 
j
o
i
n
 
c
a
n
n
o
t
 
f
a
n
 
o
u
t
.

l
i
n
e
s
_
w
i
t
h
_
h
e
a
d
e
r
 
=
 
l
i
n
e
s
.
j
o
i
n
(

 
o
r
d
e
r
s
.
s
e
l
e
c
t
(
"
s
a
l
e
s
_
o
r
d
e
r
_
i
d
"
,
 
"
r
e
t
a
i
l
e
r
_
i
d
"
,
 
"
o
r
d
e
r
_
d
a
t
e
"
,

 
"
p
r
o
m
i
s
e
d
_
d
a
t
e
"
,
 
"
f
u
l
f
i
l
l
i
n
g
_
p
l
a
n
t
_
i
d
"
)
,

 
"
s
a
l
e
s
_
o
r
d
e
r
_
i
d
"
,
 
"
l
e
f
t
"
)


l
i
n
e
s
_
b
e
f
o
r
e
 
=
 
l
i
n
e
s
_
w
i
t
h
_
h
e
a
d
e
r
.
c
o
u
n
t
(
)

l
i
n
e
s
_
e
n
r
i
c
h
e
d
 
=
 
l
i
n
e
s
_
w
i
t
h
_
h
e
a
d
e
r
.
j
o
i
n
(

 
s
h
i
p
_
b
y
_
l
i
n
e
,
 
[
"
s
a
l
e
s
_
o
r
d
e
r
_
l
i
n
e
_
i
d
"
,
 
"
s
a
l
e
s
_
o
r
d
e
r
_
i
d
"
]
,
 
"
l
e
f
t
"
)

l
i
n
e
s
_
a
f
t
e
r
 
=
 
l
i
n
e
s
_
e
n
r
i
c
h
e
d
.
c
o
u
n
t
(
)


T
h
e
 
a
s
s
e
r
t
i
o
n
 
I
S
 
t
h
e
 
l
e
s
s
o
n
:
 
l
e
f
t
-
j
o
i
n
i
n
g
 
a
n
 
a
g
g
r
e
g
a
t
e
d
 
t
a
b
l
e
 
c
a
n
n
o
t
 
a
d
d
 
r
o
w
s
.

a
s
s
e
r
t
 
l
i
n
e
s
_
a
f
t
e
r
 
=
=
 
l
i
n
e
s
_
b
e
f
o
r
e
,
 
\

 
f
"
F
A
N
-
O
U
T
:
 
j
o
i
n
 
c
h
a
n
g
e
d
 
r
o
w
 
c
o
u
n
t
 
{
l
i
n
e
s
_
b
e
f
o
r
e
}
 
-
>
 
{
l
i
n
e
s
_
a
f
t
e
r
}
"

p
r
i
n
t
(
f
"
a
n
t
i
-
f
a
n
-
o
u
t
 
c
h
e
c
k
 
p
a
s
s
e
d
:
 
{
l
i
n
e
s
_
b
e
f
o
r
e
}
 
o
r
d
e
r
 
l
i
n
e
s
 
i
n
,
 
{
l
i
n
e
s
_
a
f
t
e
r
}
 
o
u
t
"
)

w
r
i
t
e
_
d
e
l
t
a
(
l
i
n
e
s
_
e
n
r
i
c
h
e
d
,
 
"
s
i
l
v
e
r
_
s
a
l
e
s
_
o
r
d
e
r
_
l
i
n
e
s
_
e
n
r
i
c
h
e
d
"
)


In [ ]:
i
f
 
_
q
u
a
r
a
n
t
i
n
e
_
r
o
w
s
:

 
q
 
=
 
_
q
u
a
r
a
n
t
i
n
e
_
r
o
w
s
[
0
]

 
f
o
r
 
e
x
t
r
a
 
i
n
 
_
q
u
a
r
a
n
t
i
n
e
_
r
o
w
s
[
1
:
]
:

 
q
 
=
 
q
.
u
n
i
o
n
B
y
N
a
m
e
(
e
x
t
r
a
,
 
a
l
l
o
w
M
i
s
s
i
n
g
C
o
l
u
m
n
s
=
T
r
u
e
)

 
w
r
i
t
e
_
d
e
l
t
a
(
q
,
 
"
s
i
l
v
e
r
_
q
u
a
r
a
n
t
i
n
e
"
,
 
m
o
d
e
=
"
a
p
p
e
n
d
"
)

 
n
_
q
 
=
 
q
.
c
o
u
n
t
(
)

 
p
r
i
n
t
(
q
.
g
r
o
u
p
B
y
(
"
_
q
u
a
r
a
n
t
i
n
e
_
r
e
a
s
o
n
"
)
.
c
o
u
n
t
(
)
.
t
o
P
a
n
d
a
s
(
)
.
t
o
_
s
t
r
i
n
g
(
i
n
d
e
x
=
F
a
l
s
e
)
)

 
r
a
i
s
e
 
R
u
n
t
i
m
e
E
r
r
o
r
(
f
"
{
n
_
q
}
 
r
o
w
s
 
q
u
a
r
a
n
t
i
n
e
d
 
i
n
 
S
i
l
v
e
r
 
—
 
s
e
e
 
s
i
l
v
e
r
_
q
u
a
r
a
n
t
i
n
e
"
)

p
r
i
n
t
(
"
S
i
l
v
e
r
 
c
o
m
p
l
e
t
e
:
 
n
o
 
q
u
a
r
a
n
t
i
n
e
d
 
r
o
w
s
"
)
